In [1]:
# ==============================================================================
# Pattern Recognition Mini Project: K-Nearest Neighbor Classification
# Diagnostic Pipeline over Breast Cancer Diagnostic Profiles
# File: knn_classification.py
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Establish systematic visualization tracking
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
np.random.seed(42)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ------------------------------------------------------------------------------
# Task 1 & 2: Load, Parse, and Explore Dataset
# ------------------------------------------------------------------------------
print("=== 1. Data Ingestion & Cleansing ===")
file_path = "/content/drive/MyDrive/breast-cancer-body-clean-1.csv"

try:
    # Read raw data skipping the corrupted merged column row
    df_raw = pd.read_csv(file_path, skiprows=1, header=None)

    # Manually re-assign clean explicit feature metrics extracted from header string
    clean_columns = [
        "id", "diagnosis", "radius_mean", "texture_mean", "perimeter_mean",
        "area_mean", "smoothness_mean", "compactness_mean", "concavity_mean",
        "concavity_worst", "concave_points_worst", "symmetry_worst", "fractal_dimension_worst"
    ]
    df_raw.columns = clean_columns

    # Safely handle missing profile entries by dropping row entries
    df = df_raw.dropna().copy()
    print(f"Ingested Dataset Scale: {df.shape[0]} rows | Features Available: {df.shape[1]}")
    print("\n--- Missing Values Check ---")
    print(df.isnull().sum())
except Exception as e:
    print(f"Parsing Alert: Clean ingestion failed. Details: {e}")
    exit()

# Target profile transformation: Map Diagnosis (M=Malignant=1, B=Benign=0)
df['target'] = df['diagnosis'].map({'M': 1, 'B': 0})

# Isolate features matrix and label vector
X = df.drop(columns=['id', 'diagnosis', 'target'])
y = df['target']

print("\n--- Target Variable Distribution Balance ---")
print(df['diagnosis'].value_counts(normalize=True))

# Visualize feature metrics distributions
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(data=df, x='radius_mean', hue='diagnosis', multiple='stack', palette='crest', kde=True)
plt.title('Distribution Profile: Radius Mean by Diagnosis')

plt.subplot(1, 2, 2)
sns.scatterplot(data=df, x='radius_mean', y='texture_mean', hue='diagnosis', palette='flare', alpha=0.8)
plt.title('Feature Interaction: Radius vs. Texture Mean')
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight')
plt.close()


=== 1. Data Ingestion & Cleansing ===
Ingested Dataset Scale: 98 rows | Features Available: 13

--- Missing Values Check ---
id                         0
diagnosis                  0
radius_mean                0
texture_mean               0
perimeter_mean             0
area_mean                  0
smoothness_mean            0
compactness_mean           0
concavity_mean             0
concavity_worst            0
concave_points_worst       0
symmetry_worst             0
fractal_dimension_worst    0
dtype: int64

--- Target Variable Distribution Balance ---
diagnosis
M    0.642857
B    0.357143
Name: proportion, dtype: float64


In [4]:
# ------------------------------------------------------------------------------
# Task 3 & 4: Train/Test Splitting & Feature Standardization
# ------------------------------------------------------------------------------
print("\n=== 2. Stratified Partitioning & Z-Score Scaling ===")
# 80/20 train/test split with stratification to preserve target balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training Matrix Scale: {X_train.shape[0]} rows | Evaluation Scale: {X_test.shape[0]} rows")

# Apply StandardScaler (Critical for distance-based calculations like KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



=== 2. Stratified Partitioning & Z-Score Scaling ===
Training Matrix Scale: 78 rows | Evaluation Scale: 20 rows


In [5]:
# ------------------------------------------------------------------------------
# Task 5, 6 & 7: Hyperparameter Tuning via Cross-Validation
# ------------------------------------------------------------------------------
print("\n=== 3. K-Parameter Optimization via 5-Fold Cross-Validation ===")
k_values = [1, 3, 5, 7, 9, 11]
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    # Using 5-fold cross-validation over scaled training space
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())
    print(f"Parameter k = {k:02d} | Mean Cross-Validation Accuracy: {scores.mean():.4f}")

# Find the optimal k value
optimal_k = k_values[np.argmax(cv_scores)]
print(f"\n[OPTIMIZATION DECISION]: Peak accuracy achieved at k = {optimal_k}")

# Plot k parameter vs. Cross-Validated Accuracy
plt.figure(figsize=(8, 5))
plt.plot(k_values, cv_scores, marker='o', linestyle='--', color='darkblue', linewidth=2)
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Mean CV Accuracy')
plt.title('KNN Hyperparameter Tuning Grid')
plt.axvline(x=optimal_k, color='crimson', linestyle=':', label=f'Optimal k ({optimal_k})')
plt.legend()
plt.savefig('knn_tuning_curve.png', bbox_inches='tight')
plt.close()



=== 3. K-Parameter Optimization via 5-Fold Cross-Validation ===
Parameter k = 01 | Mean Cross-Validation Accuracy: 0.9367
Parameter k = 03 | Mean Cross-Validation Accuracy: 0.9367
Parameter k = 05 | Mean Cross-Validation Accuracy: 0.9367
Parameter k = 07 | Mean Cross-Validation Accuracy: 0.9367
Parameter k = 09 | Mean Cross-Validation Accuracy: 0.9367
Parameter k = 11 | Mean Cross-Validation Accuracy: 0.9492

[OPTIMIZATION DECISION]: Peak accuracy achieved at k = 11


In [6]:
# ------------------------------------------------------------------------------
# Task 8: Evaluate Final Model Predictions
# ------------------------------------------------------------------------------
print("\n=== 4. Comprehensive Evaluation Metrics Dashboard ===")
# Fit final optimized classifier
final_knn = KNeighborsClassifier(n_neighbors=optimal_k)
final_knn.fit(X_train_scaled, y_train)

# Generate final predictions
y_pred = final_knn.predict(X_test_scaled)

print(f"Final Out-Of-Sample Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("--- Detailed Classification Matrix Report ---")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Malignant (1)']))

# Confusion Matrix Heatmap Representation
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.ylabel('Ground Truth Profile')
plt.xlabel('Classifier Prediction Output')
plt.title(f'Confusion Matrix Heatmap (k={optimal_k})')
plt.tight_layout()
plt.savefig('knn_confusion_matrix.png', bbox_inches='tight')
plt.close()



=== 4. Comprehensive Evaluation Metrics Dashboard ===
Final Out-Of-Sample Accuracy Score: 0.8000

--- Detailed Classification Matrix Report ---
               precision    recall  f1-score   support

   Benign (0)       0.80      0.57      0.67         7
Malignant (1)       0.80      0.92      0.86        13

     accuracy                           0.80        20
    macro avg       0.80      0.75      0.76        20
 weighted avg       0.80      0.80      0.79        20



In [7]:
# ------------------------------------------------------------------------------
# Task 9: Inference Pipeline Check over Unseen Entry Samples
# ------------------------------------------------------------------------------
print("\n=== 5. In-Production Synthetic Inference Tests ===")
# Simulating a new anonymous measurement vector using localized means
unseen_sample = np.array([X_train.mean(axis=0)])

# Scale using the established training parameters
unseen_sample_scaled = scaler.transform(unseen_sample)
prediction = final_knn.predict(unseen_sample_scaled)
probability = final_knn.predict_proba(unseen_sample_scaled)[0]

print(f"Unseen Vector Classification Result: {'Malignant (1)' if prediction[0] == 1 else 'Benign (0)'}")
print(f"Calculated Label Probabilities: [Benign: {probability[0]:.2f}, Malignant: {probability[1]:.2f}]")



=== 5. In-Production Synthetic Inference Tests ===
Unseen Vector Classification Result: Malignant (1)
Calculated Label Probabilities: [Benign: 0.00, Malignant: 1.00]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
